In [2]:
!pip install requests==2.32.4


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 6.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.34.2
    Uninstalling requests-2.34.2:
      Successfully uninstalled requests-2.34.2


In [3]:
# Install Required Libraries
!pip -q install langchain
!pip -q install langchain-community
!pip -q install sentence-transformers
!pip -q install transformers
!pip -q install accelerate
!pip -q install faiss-cpu
!pip -q install pypdf
!pip -q install gradio
!pip -q install rank-bm25
!pip -q install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 26.4 MB/s eta 0:00:00


In [4]:
import langchain
import sentence_transformers
import transformers
import faiss
import gradio
import rank_bm25
import datasets

print("All libraries imported successfully!")

All libraries imported successfully!


In [5]:
print("="*60)
print("Retrieval-Augmented Generation (RAG)")
print("Week 7 Assignment")
print("="*60)

print("\nPipeline")

print("""
1. Document Ingestion
2. Text Chunking
3. Embedding Generation
4. FAISS Vector Database
5. Query Embedding
6. Context Retrieval
7. Answer Generation
8. Optimization
""")

Retrieval-Augmented Generation (RAG)
Week 7 Assignment

Pipeline

1. Document Ingestion
2. Text Chunking
3. Embedding Generation
4. FAISS Vector Database
5. Query Embedding
6. Context Retrieval
7. Answer Generation
8. Optimization



In [6]:

# Upload PDF or TXT File

from google.colab import files

uploaded = files.upload()

file_name = list(uploaded.keys())[0]

print("Uploaded File:", file_name)

Saving Sample_Ai_Book.pdf to Sample_Ai_Book.pdf
Uploaded File: Sample_Ai_Book.pdf


In [7]:
# Load PDF or TXT Document
from langchain_community.document_loaders import PyPDFLoader, TextLoader

if file_name.endswith(".pdf"):
    loader = PyPDFLoader(file_name)

elif file_name.endswith(".txt"):
    loader = TextLoader(file_name)

else:
    raise Exception("Only PDF and TXT files are supported.")

documents = loader.load()

print("Number of Pages/Documents:", len(documents))

/tmp/ipykernel_967/488927711.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, TextLoader


Number of Pages/Documents: 44


In [8]:
# Display First Page
print(documents[0].page_content[:1000])

INTRODUCTION TO AI
World Travel & Tourism Council
< Contents  | 1
INTRODUCTION 
TO ARTIFICIAL 
INTELLIGENCE (AI) 
TECHNOLOGY
GUIDE FOR TRAVEL & TOURISM LEADERS
January 2024


In [10]:
import warnings
warnings.filterwarnings('ignore')

!pip -q install langchain-text-splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(

    chunk_size=600,
    chunk_overlap=100

)

chunks = text_splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 207


In [11]:
# Show Sample Chunks

for i in range(min(3, len(chunks))):

    print("="*70)

    print("Chunk", i+1)

    print(chunks[i].page_content[:500])

Chunk 1
INTRODUCTION TO AI
World Travel & Tourism Council
< Contents  | 1
INTRODUCTION 
TO ARTIFICIAL 
INTELLIGENCE (AI) 
TECHNOLOGY
GUIDE FOR TRAVEL & TOURISM LEADERS
January 2024
Chunk 2
INTRODUCTION TO AI
World Travel & Tourism Council
2
< Contents  |
CONTENTS
FOREWORD ...........................................................................................................3
INTRODUCTION ..................................................................................................4
Brief history of Artificial Intelligence (AI) 4
What is Artificial Intelligence & Why There is Global Interest Now  7
Algorithms : The Brains of AI 8
Data : The Fuel That Drives AI 12
Computing 
Chunk 3
Computing Power : The Machines Behind AI 16
Types of Artificial Intelligence 21
Generative AI 24
Global Digital Divide & AI Skills Gap 31
QUIZ....................................................................................................................... 34
ANNEX : AI TERMINOLOGY...............

In [12]:
# Load Embedding Model

from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(

    model_name="sentence-transformers/all-MiniLM-L6-v2"

)

print("Embedding model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


In [13]:
# Create FAISS Vector Database

from langchain_community.vectorstores import FAISS

vector_db = FAISS.from_documents(

    chunks,
    embedding_model

)

print("Vector database created successfully.")

Vector database created successfully.


In [18]:
retriever = vector_db.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k":3,
        "fetch_k":10,
        "lambda_mult":0.7
    }
)

print("MMR Retriever Ready")

MMR Retriever Ready


In [19]:
# Ask Question

question = input("Enter your question: ")

retrieved_docs = retriever.invoke(question)

print("Retrieved Chunks:", len(retrieved_docs))

Enter your question: What is Artificial Intelligence?
Retrieved Chunks: 3


In [30]:
# Display Retrieved Context
context = ""

print("="*80)
print("Retrieved Chunks")
print("="*80)

for i, doc in enumerate(retrieved_docs):

    page = doc.metadata.get("page", "Unknown")

    print(f"\n Chunk {i+1} (Page {page+1 if page!='Unknown' else page})")
    print("-"*80)
    print(doc.page_content)

    context += doc.page_content + "\n\n"

Retrieved Chunks

 Chunk 1 (Page 7)
--------------------------------------------------------------------------------
WHAT IS ARTIFICIAL INTELLIGENCE & WHY THERE IS GLOBAL INTEREST NOW 
One of the most important aspects of AI is that it is a multi-use technology. Like electricity it can be applied 
in lots of different ways, to lots of different scenarios.
There is no single, universally accepted definition for Artificial Intelligence, but the Oxford English Dictionary 
defines AI as “the capacity of computers, or other machines, to exhibit intelligent behaviour”. This means AI 
systems appear to think, learn and act like humans and in some cases exceed the capabilities of humans. AI

 Chunk 2 (Page 4)
--------------------------------------------------------------------------------
BRIEF HISTORY OF ARTIFICIAL INTELLIGENCE (AI)
AI has gained significant attention in recent years – and especially in 2023 – but AI is not new and can trace its 
history back to the development of computers a

In [31]:
# Load LLM
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"

model.to(device)

print("Model loaded successfully!")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model loaded successfully!


In [32]:
prompt = f"""
You are an expert AI assistant.

Your task is to answer the user's question ONLY using the information provided in the retrieved context.

Instructions:
1. Read the context carefully.
2. Provide a complete and well-structured answer in 4-6 sentences.
3. Do NOT use outside knowledge.
4. Do NOT copy the context word-for-word.
5. If the answer is not available in the context, reply exactly:
"I could not find the answer in the uploaded document."

-------------------------
Retrieved Context:
{context}
-------------------------

Question:
{question}

Answer:
"""

# Tokenize input
inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=1024
).to(device)

# Generate answer
outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=False,
    num_beams=4,
    early_stopping=True
)

# Decode answer
answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("=" * 80)
print("Generated Answer")
print("=" * 80)
print(answer)

Generated Answer
The capacity of computers, or other machines, to exhibit intelligent behaviour


In [34]:
# Build BM25 Keyword Index

from rank_bm25 import BM25Okapi

# Tokenize each chunk
tokenized_chunks = [
    doc.page_content.lower().split()
    for doc in chunks
]

# Build BM25 Index
bm25 = BM25Okapi(tokenized_chunks)

print("="*70)
print("BM25 Index Created Successfully")
print("Total Documents Indexed:", len(tokenized_chunks))

BM25 Index Created Successfully
Total Documents Indexed: 207


In [37]:

# Hybrid Retrieval

import numpy as np

def hybrid_retrieve(query, top_k=3):

    faiss_docs = retriever.invoke(query)

    tokenized_query = query.lower().split()

    scores = bm25.get_scores(tokenized_query)

    bm25_indices = np.argsort(scores)[::-1][:top_k]

    bm25_docs = [chunks[i] for i in bm25_indices]


    merged = []

    seen = set()

    for doc in faiss_docs + bm25_docs:

        if doc.page_content not in seen:

            merged.append(doc)

            seen.add(doc.page_content)

    return merged[:top_k]

In [38]:
question = input("Ask your question : ")

retrieved_docs = hybrid_retrieve(question)

print("="*80)

print("Retrieved Documents")

print("="*80)

context = ""

for i, doc in enumerate(retrieved_docs):

    print(f"\nChunk {i+1}")

    print("-"*80)

    print(doc.page_content)

    context += doc.page_content + "\n"

Ask your question : What is Artificial Intelligence?
Retrieved Documents

Chunk 1
--------------------------------------------------------------------------------
WHAT IS ARTIFICIAL INTELLIGENCE & WHY THERE IS GLOBAL INTEREST NOW 
One of the most important aspects of AI is that it is a multi-use technology. Like electricity it can be applied 
in lots of different ways, to lots of different scenarios.
There is no single, universally accepted definition for Artificial Intelligence, but the Oxford English Dictionary 
defines AI as “the capacity of computers, or other machines, to exhibit intelligent behaviour”. This means AI 
systems appear to think, learn and act like humans and in some cases exceed the capabilities of humans. AI

Chunk 2
--------------------------------------------------------------------------------
BRIEF HISTORY OF ARTIFICIAL INTELLIGENCE (AI)
AI has gained significant attention in recent years – and especially in 2023 – but AI is not new and can trace its 
history ba

In [39]:
prompt = f"""
You are an expert AI assistant.

Answer ONLY using the retrieved context.

If the answer is unavailable,
say

"I could not find this information in the uploaded document."

Context:
{context}

Question:
{question}

Answer:
"""

inputs = tokenizer(

    prompt,

    return_tensors="pt",

    truncation=True,

    max_length=1024

).to(device)

outputs = model.generate(

    **inputs,

    max_new_tokens=200,

    do_sample=False,

    num_beams=4

)

answer = tokenizer.decode(

    outputs[0],

    skip_special_tokens=True

)

print("="*80)

print(answer)

There is no single, universally accepted definition for Artificial Intelligence, but the Oxford English Dictionary defines AI as “the capacity of computers, or other machines, to exhibit intelligent behaviour”. This means AI systems appear to think, learn and act like humans and in some cases exceed the capabilities of humans.


In [50]:
# Experiment 1 : Chunk Size Comparison

!pip -q install langchain-text-splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunk_sizes = [300, 600, 900]

print("="*90)
print("EXPERIMENT 1 : CHUNK SIZE COMPARISON")
print("="*90)

for size in chunk_sizes:

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=int(size*0.2)
    )

    split_docs = splitter.split_documents(documents)

    print(f"\nChunk Size : {size}")

    print(f"Chunk Overlap : {int(size*0.2)}")

    print(f"Total Chunks : {len(split_docs)}")

EXPERIMENT 1 : CHUNK SIZE COMPARISON

Chunk Size : 300
Chunk Overlap : 60
Total Chunks : 426

Chunk Size : 600
Chunk Overlap : 120
Total Chunks : 229

Chunk Size : 900
Chunk Overlap : 180
Total Chunks : 151


In [51]:
# Experiment 2 : Top-K Comparison

question = "What is Artificial Intelligence?"

top_k_values = [3,5,8]

print("="*90)
print("EXPERIMENT 2 : TOP-K RETRIEVAL")
print("="*90)

for k in top_k_values:

    retriever = vector_db.as_retriever(
        search_kwargs={"k":k}
    )

    docs = retriever.invoke(question)

    print(f"\nTop K = {k}")

    print("Retrieved Chunks :",len(docs))

    print("First Chunk Preview:")

    print(docs[0].page_content[:300])

    print("-"*80)

EXPERIMENT 2 : TOP-K RETRIEVAL

Top K = 3
Retrieved Chunks : 3
First Chunk Preview:
WHAT IS ARTIFICIAL INTELLIGENCE & WHY THERE IS GLOBAL INTEREST NOW 
One of the most important aspects of AI is that it is a multi-use technology. Like electricity it can be applied 
in lots of different ways, to lots of different scenarios.
There is no single, universally accepted definition for Art
--------------------------------------------------------------------------------

Top K = 5
Retrieved Chunks : 5
First Chunk Preview:
WHAT IS ARTIFICIAL INTELLIGENCE & WHY THERE IS GLOBAL INTEREST NOW 
One of the most important aspects of AI is that it is a multi-use technology. Like electricity it can be applied 
in lots of different ways, to lots of different scenarios.
There is no single, universally accepted definition for Art
--------------------------------------------------------------------------------

Top K = 8
Retrieved Chunks : 8
First Chunk Preview:
WHAT IS ARTIFICIAL INTELLIGENCE & WHY THERE IS

In [52]:
# Experiment 3 : Retrieval Comparison

question = "What is Artificial Intelligence?"

print("="*90)
print("EXPERIMENT 3 : FAISS vs HYBRID")
print("="*90)

# -------- FAISS --------

faiss_docs = retriever.invoke(question)

print("\nFAISS Retrieval")

for i,doc in enumerate(faiss_docs[:3]):

    print(f"\nChunk {i+1}")

    print(doc.page_content[:250])

# -------- Hybrid --------

hybrid_docs = hybrid_retrieve(question)

print("\n\nHYBRID Retrieval")

for i,doc in enumerate(hybrid_docs):

    print(f"\nChunk {i+1}")

    print(doc.page_content[:250])

EXPERIMENT 3 : FAISS vs HYBRID

FAISS Retrieval

Chunk 1
WHAT IS ARTIFICIAL INTELLIGENCE & WHY THERE IS GLOBAL INTEREST NOW 
One of the most important aspects of AI is that it is a multi-use technology. Like electricity it can be applied 
in lots of different ways, to lots of different scenarios.
There is 

Chunk 2
BRIEF HISTORY OF ARTIFICIAL INTELLIGENCE (AI)
AI has gained significant attention in recent years – and especially in 2023 – but AI is not new and can trace its 
history back to the development of computers after the Second World War, with the Dartmo

Chunk 3
the way they learn.  This includes machine learning and deep learning, with different techniques for training 
these AI systems including supervised learning, unsupervised learning and reinforcement learning.
The second useful way to group, or classi


HYBRID Retrieval

Chunk 1
WHAT IS ARTIFICIAL INTELLIGENCE & WHY THERE IS GLOBAL INTEREST NOW 
One of the most important aspects of AI is that it is a multi-use technology

In [53]:
# Experiment 4 : Retrieval Time

import time

question = "What is Artificial Intelligence?"

print("="*90)
print("EXPERIMENT 4 : RETRIEVAL TIME")
print("="*90)

start = time.time()

docs = hybrid_retrieve(question)

retrieval_time = time.time()-start

print("Retrieved Chunks :",len(docs))

print(f"Retrieval Time : {retrieval_time:.4f} seconds")

EXPERIMENT 4 : RETRIEVAL TIME
Retrieved Chunks : 3
Retrieval Time : 0.0468 seconds


In [54]:
# Experiment 5 : Generation Time

context=""

for doc in docs:

    context += doc.page_content + "\n"

prompt=f"""

Answer only from context.

Context:

{context}

Question:

{question}

Answer:

"""

inputs=tokenizer(

    prompt,

    return_tensors="pt",

    truncation=True,

    max_length=1024

).to(device)

start=time.time()

outputs=model.generate(

    **inputs,

    max_new_tokens=150,

    do_sample=False

)

generation_time=time.time()-start

answer=tokenizer.decode(

    outputs[0],

    skip_special_tokens=True

)

print("="*90)

print("EXPERIMENT 5 : GENERATION TIME")

print("="*90)

print(answer)

print()

print(f"Generation Time : {generation_time:.4f} seconds")

EXPERIMENT 5 : GENERATION TIME
the capacity of computers, or other machines, to exhibit intelligent behaviour

Generation Time : 0.4378 seconds


In [55]:
# Experiment Summary

import pandas as pd

summary = pd.DataFrame({

    "Experiment":[

        "Chunk Size",

        "Top-K Retrieval",

        "Hybrid Retrieval",

        "Retrieval Time",

        "Generation Time"

    ],

    "Observation":[

        "Compared 300,600,900",

        "Compared Top3,Top5,Top8",

        "Compared FAISS vs Hybrid",

        f"{retrieval_time:.4f} sec",

        f"{generation_time:.4f} sec"

    ]

})

summary

,Experiment,Observation
0,Chunk Size,"Compared 300,600,900"
1,Top-K Retrieval,"Compared Top3,Top5,Top8"
2,Hybrid Retrieval,Compared FAISS vs Hybrid
3,Retrieval Time,0.0468 sec
4,Generation Time,0.4378 sec


In [56]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

query = "What is Artificial Intelligence?"

docs = hybrid_retrieve(query, top_k=5)

pairs = [[query, doc.page_content] for doc in docs]

scores = reranker.predict(pairs)

ranked = sorted(
    zip(scores, docs),
    reverse=True,
    key=lambda x: x[0]
)

print("="*90)
print("RE-RANKED DOCUMENTS")
print("="*90)

for score, doc in ranked:

    print(f"\nScore : {score:.4f}")

    print(doc.page_content[:350])

    print("-"*80)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

RE-RANKED DOCUMENTS

Score : 8.3278
WHAT IS ARTIFICIAL INTELLIGENCE & WHY THERE IS GLOBAL INTEREST NOW 
One of the most important aspects of AI is that it is a multi-use technology. Like electricity it can be applied 
in lots of different ways, to lots of different scenarios.
There is no single, universally accepted definition for Artificial Intelligence, but the Oxford English Dicti
--------------------------------------------------------------------------------

Score : 6.1214
BRIEF HISTORY OF ARTIFICIAL INTELLIGENCE (AI)
AI has gained significant attention in recent years – and especially in 2023 – but AI is not new and can trace its 
history back to the development of computers after the Second World War, with the Dartmouth Conference in 
1956 bringing together researchers from multiple fields to explore “thinking mach
--------------------------------------------------------------------------------

Score : 3.1667
play chess.
Business leaders could use narrow AI 
systems to automat

In [57]:
# Final RAG Chat Function

def rag_chat(question):

    # Retrieve documents
    docs = hybrid_retrieve(question, top_k=5)

    # Re-rank documents
    pairs = [[question, doc.page_content] for doc in docs]
    scores = reranker.predict(pairs)

    ranked_docs = sorted(
        zip(scores, docs),
        reverse=True,
        key=lambda x: x[0]
    )

    # Build context
    context = ""

    for score, doc in ranked_docs[:3]:
        context += doc.page_content + "\n\n"

    prompt = f"""
You are an expert AI assistant.

Answer ONLY using the retrieved context.

If the answer is not present in the context, say:

"I could not find this information in the uploaded document."

Context:
{context}

Question:
{question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        num_beams=4,
        do_sample=False
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [58]:
# Test Complete Pipeline

question = input("Ask a question from the uploaded document: ")

print("\nGenerating Answer...\n")

answer = rag_chat(question)

print("="*90)

print(answer)

Ask a question from the uploaded document: What is Machine Learning?

Generating Answer...

a type of AI that involves training algorithms with large amounts of data


In [59]:
# Gradio Interface

import gradio as gr

demo = gr.Interface(
    fn=rag_chat,

    inputs=gr.Textbox(
        lines=2,
        label="Question",
        placeholder="Ask a question about your uploaded document..."
    ),

    outputs=gr.Textbox(
        lines=8,
        label="Generated Answer"
    ),

    title="Retrieval-Augmented Generation (RAG)",

    description="""
Upload a document, generate embeddings,
retrieve relevant information,
and answer questions using Retrieval-Augmented Generation.
"""
)

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d4c26cd45e09d8b3c6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://d4c26cd45e09d8b3c6.gradio.live


In [60]:
# Evaluation

test_questions = [

    "What is Artificial Intelligence?",

    "Explain Machine Learning.",

    "What is Deep Learning?",

    "Who introduced the Turing Test?",

    "What are the three core components of AI?"

]

for q in test_questions:

    print("="*100)

    print("Question:")

    print(q)

    print()

    print("Answer:")

    print(rag_chat(q))

    print()

Question:
What is Artificial Intelligence?

Answer:
a multi-use technology. Like electricity it can be applied in lots of different ways, to lots of different scenarios. There is no single, universally accepted definition for Artificial Intelligence, but the Oxford English Dictionary defines AI as “the capacity of computers, or other machines, to exhibit intelligent behaviour”. This means AI systems appear to think, learn and act like humans and in some cases exceed the capabilities of humans.

Question:
Explain Machine Learning.

Answer:
Machine learning is a type of AI that involves training algorithms with large amounts of data, so that a computer is capable of making predictions, or decisions. An email spam filter can be an example of an AI system using machine learning. By training an AI algorithm on a dataset of known spam and non-spam emails, it can learn to distinguish between the two and is then able to automatically detect and filter out new spam messages, even though it has 

In [61]:
# Final Report

print("="*90)

print("RAG SYSTEM REPORT")

print("="*90)

print(f"Document Pages       : {len(documents)}")

print(f"Document Chunks      : {len(chunks)}")

print("Embedding Model      : all-MiniLM-L6-v2")

print("Vector Database      : FAISS")

print("Keyword Search       : BM25")

print("Hybrid Retrieval     : Enabled")

print("Cross Encoder        : Enabled")

print("Language Model       : google/flan-t5-base")

print(f"Retrieval Time       : {retrieval_time:.4f} sec")

print(f"Generation Time      : {generation_time:.4f} sec")

RAG SYSTEM REPORT
Document Pages       : 44
Document Chunks      : 207
Embedding Model      : all-MiniLM-L6-v2
Vector Database      : FAISS
Keyword Search       : BM25
Hybrid Retrieval     : Enabled
Cross Encoder        : Enabled
Language Model       : google/flan-t5-base
Retrieval Time       : 0.0468 sec
Generation Time      : 0.4378 sec


# Observations

During the implementation of the Retrieval-Augmented Generation (RAG) system, several observations were made:

- The document ingestion module successfully processed PDF documents and extracted the text content.
- Recursive text chunking improved retrieval efficiency by dividing large documents into manageable chunks.
- The **sentence-transformers/all-MiniLM-L6-v2** embedding model generated meaningful vector representations for semantic search.
- FAISS provided fast similarity search over embedded document chunks.
- BM25 keyword retrieval complemented semantic retrieval by improving results for exact keyword matches.
- Hybrid Retrieval (FAISS + BM25) produced more relevant document chunks than using semantic retrieval alone.
- Cross-Encoder re-ranking further improved the ordering of retrieved chunks, resulting in more contextually relevant information being passed to the language model.
- The language model generated grounded responses based on the retrieved context instead of relying solely on its internal knowledge.
- Experimental comparisons showed that retrieval quality is affected by chunk size, retrieval strategy, and the number of retrieved chunks (Top-K).
- The Gradio interface provided an easy-to-use interactive environment for document-based question answering.

#Conclusion

A complete Retrieval-Augmented Generation (RAG) system was successfully developed for answering questions from custom documents.

The project implemented all major stages of a RAG pipeline, including document ingestion, text preprocessing, chunking, embedding generation, vector database creation, query embedding, document retrieval, and grounded response generation.

Several optimization techniques were explored, including chunk size comparison, hybrid retrieval using FAISS and BM25, and Cross-Encoder re-ranking. These experiments demonstrated that retrieval quality can be improved through better indexing and ranking strategies.

Overall, the system provides accurate, context-aware responses while reducing hallucinations by grounding answers in the uploaded documents. This project demonstrates how modern RAG architectures combine retrieval and language models to build reliable document question-answering systems suitable for knowledge assistants, enterprise search, and AI-powered documentation tools.